# Paper results: recall by query distance and attribute count

Every plot here reads `<run_dir>/preds/` — the top-100 rankings `paper.sh` writes for each
condition — so the whole notebook runs on **CPU** and never re-encodes a model. That means it
is safe to run while `paper.sh` is training on the GPUs.

The condition list is parsed out of `paper.sh`'s own `CONDITIONS` table, so adding a row there
is all it takes for a condition to appear below.

**Plots**
1. **V ablation** — recall vs. the distance normalizer `V`, on synthetic queries, per modality.
2. **Ours vs. baselines** — one figure per (modality, query kind), Recall@{1,5,50} against both
   query distance and attribute count. `ours-mse` is pinned to **V=40**, the value the main grid uses.

**Covariates**
- `query_distance` comes from `triplets.jsonl`. It is per triplet row, not per query: each query has
  one row against a hard/substitute negative carrying the real distance and one against a random
  negative carrying a `-1` sentinel. The sentinel rows are dropped and the rest averaged per query.
- `num_attributes` is parsed from the rendered query text. Checked against the image dataset's exact
  `selected_*` feature lists it agrees on 2995/3000 sampled rows. Text `original` queries are bare
  keywords (`'fv relay'`) with no attributes to parse, so those counts are borrowed from the same
  rows' synthetic run — an alignment the loader asserts on product ids rather than assumes.

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from utils.paper_analysis import (
    parse_conditions, discover_runs, match_conditions, health_check, load_all,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

NOTE = "paper"
MODELS_ROOT = "models"
KS = (1, 5, 20)          # the recall cutoffs the paper reports
MAIN_V = 40              # ours-mse is pinned to V=40 everywhere outside the ablation
MIN_BIN = 30             # drop covariate bins backed by fewer queries than this
FIG_DIR = "paper/figs"
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Conditions and health

`health_check` is the gate: a condition is only plotted if it actually produced usable predictions.
A run whose queries came back **blank** collapses to a single empty anchor and scores ~0 — that is
not a result, it is a broken inference job, and it is excluded rather than drawn as a flat line.

In [ ]:
conditions = parse_conditions("paper.sh")
runs = discover_runs(MODELS_ROOT, note=NOTE)
matched = match_conditions(conditions, runs)
health = health_check(matched)

print(f"{len(conditions)} conditions in paper.sh | "
      f"{int(health['healthy'].sum())} usable | {int((~health['healthy']).sum())} not")
display(health[~health["healthy"]][
    ["modality", "style", "query_kind", "V", "n_queries", "problem"]])

In [ ]:
# health_check emits one row per matched condition, in order, so mask by position.
usable = matched[health["healthy"].to_numpy()].reset_index(drop=True)
display(usable[["modality", "style", "query_kind", "V"]]
        .sort_values(["modality", "query_kind", "style", "V"]))

## 2. Load per-query results

One row per (condition, query): recall at each cutoff plus the two covariates.

In [ ]:
# Text `original` queries carry no attribute list; borrow it from the synthetic run of the
# same rows. load_all verifies the row alignment on positive_id/negative_id and raises if it fails.
attribute_reference = {}
for row in usable[(usable["modality"] == "text") & (usable["query_kind"] == "original")].itertuples():
    same_rows = usable[(usable["modality"] == "text")
                       & (usable["query_kind"] == "synthetic")
                       & (usable["style"] == row.style)
                       & (usable["V"].isna() == pd.isna(row.V))]
    if len(same_rows):
        attribute_reference[row.run_dir] = same_rows.iloc[0].run_dir

results = load_all(usable, ks=KS, attribute_reference=attribute_reference)
print(f"{len(results):,} per-query rows across {results['run_dir'].nunique()} conditions")
results.head()

In [ ]:
STYLE_ORDER = ["untrained", "baseline-triplet", "infonce", "cosent", "classic-mse", "ours-mse"]
STYLE_LABEL = {"untrained": "Untrained", "baseline-triplet": "Triplet", "infonce": "InfoNCE",
               "cosent": "CoSENT", "classic-mse": "Classic MSE", "ours-mse": "Ours (MSE)"}


def binned(frame, covariate, k, min_count=MIN_BIN):
    """Mean recall@k per integer covariate bin, dropping bins with thin support."""
    f = frame.dropna(subset=[covariate]).copy()
    f["bin"] = f[covariate].round().astype(int)
    grouped = (f.groupby(["style", "bin"])
                 .agg(**{f"recall@{k}": (f"recall@{k}", "mean"), "n": ("query_id", "size")})
                 .reset_index())
    return grouped[grouped["n"] >= min_count]

## 3. V ablation

Synthetic queries only — this sweep is what `V=40` in the main grid was selected from.
The dashed line marks the value the rest of the paper uses.

In [ ]:
ablation = results[(results["style"] == "ours-mse")
                   & (results["query_kind"] == "synthetic")
                   & results["V"].notna()]
ablation_means = (ablation.groupby(["modality", "V"])[[f"recall@{k}" for k in KS]]
                  .mean().reset_index())

modalities = sorted(ablation_means["modality"].unique())
fig, axes = plt.subplots(1, len(modalities), figsize=(5.5 * len(modalities), 4.2), squeeze=False)
for ax, modality in zip(axes[0], modalities):
    sub = ablation_means[ablation_means["modality"] == modality].sort_values("V")
    for k in KS:
        ax.plot(sub["V"], sub[f"recall@{k}"], marker="o", label=f"Recall@{k}")
    ax.axvline(MAIN_V, color="grey", linestyle="--", linewidth=1)
    ax.set_title(f"V ablation — {modality} (synthetic)")
    ax.set_xlabel("V (distance normalizer)")
    ax.set_ylabel("Recall")
    ax.set_xticks(sorted(sub["V"].unique()))
    ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "v_ablation.png"), dpi=150)
plt.show()

display(ablation_means.round(4))

## 4. Ours vs. baselines

One figure per (modality, query kind). Rows are the recall cutoffs, columns are the two covariates.
`ours-mse` is the **V=40** run; the other styles have no V.

In [ ]:
main_grid = results[results["V"].isna() | (results["V"] == MAIN_V)]
COVARIATES = [("query_distance", "Query distance (differentiating attributes)"),
              ("num_attributes", "Number of attributes in query")]

for (modality, query_kind), group in main_grid.groupby(["modality", "query_kind"]):
    order = [s for s in STYLE_ORDER if s in set(group["style"])]
    palette = dict(zip(order, sns.color_palette("colorblind", len(order))))

    fig, axes = plt.subplots(len(KS), len(COVARIATES),
                             figsize=(13, 3.6 * len(KS)), sharey="row", squeeze=False)
    drew_anything = False
    for i, k in enumerate(KS):
        for j, (covariate, xlabel) in enumerate(COVARIATES):
            ax = axes[i][j]
            curves = binned(group, covariate, k)
            if curves.empty:
                ax.set_visible(False)
                continue
            drew_anything = True
            for style in order:
                line = curves[curves["style"] == style].sort_values("bin")
                if len(line):
                    ax.plot(line["bin"], line[f"recall@{k}"], marker="o", markersize=4,
                            label=STYLE_LABEL[style], color=palette[style])
            ax.set_xlabel(xlabel)
            ax.set_ylabel(f"Recall@{k}")
            ax.set_ylim(0, 1.02)
            if i == 0 and j == 0:
                ax.legend(fontsize=8, ncol=2)
    if not drew_anything:
        plt.close(fig)
        continue
    fig.suptitle(f"{modality} / {query_kind} queries — ours (V={MAIN_V}) vs baselines",
                 fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(os.path.join(FIG_DIR, f"compare_{modality}_{query_kind}.png"), dpi=150)
    curve_data = pd.concat(
        [binned(group, covariate, k).assign(covariate=covariate, k=k)
         for k in KS for covariate, _ in COVARIATES], ignore_index=True)
    curve_data.to_csv(os.path.join(FIG_DIR, f"compare_{modality}_{query_kind}.csv"), index=False)
    plt.show()

## 5. Overall numbers

Averaged over all queries, per condition.

In [ ]:
summary = (main_grid.groupby(["modality", "query_kind", "style"])[[f"recall@{k}" for k in KS]]
           .mean().round(4).reset_index())
summary.to_csv(os.path.join(FIG_DIR, "summary.csv"), index=False)
display(summary)